# Patent analysis

In [1]:
import pandas as pd

In [136]:
# AltairSaver = altair_save_utils.AltairSaver()

In [2]:
from discovery_child_development.utils import analysis_utils as au
from discovery_child_development.utils import plotting_utils as pu

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import utils
import importlib
importlib.reload(utils);

2024-06-01 07:19:01,267 - botocore.credentials - INFO - Found credentials in environment variables.
2024-06-01 07:19:02,753 - datasets - INFO - PyTorch version 2.1.2 available.


/opt/homebrew/Caskroom/miniconda/base/envs/discovery_child_development/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load data

In [331]:
# Labelled data
data_df = utils.load_patents_data().query("topics != 'arts'")

In [332]:
len(data_df)

25972

In [333]:
# Taxonomy dataframe
topics_df = utils.load_topic_data()

In [334]:
topics_df

,topic,type,subtype,name
0,genetics,Biosciences,Genetics,Genetics
1,neuroscience,Biosciences,Neuroscience,Neuroscience
2,operations,Child care & preschool,Operations,Operations
3,preschool,Child care & preschool,Preschool,Preschool
4,cognitive,Development & learning,Cognitive development,Cognitive development
5,communication,Development & learning,Communication and language,Communication and language
6,arts,Development & learning,Expressive arts and design,Expressive arts and design
7,literacy,Development & learning,Literacy,Literacy
8,mathematics,Development & learning,Mathematics,Mathematics
9,emotional,Development & learning,Personal social emotional,Personal social emotional


In [335]:
# Transform to one id and topic pair per row
data_exploded_df = utils.explode_data(data_df).query("topic != 'arts'")

## Corrections

In [336]:
check_types = ['Child care & preschool', 'Development & learning', 'Health', 'Parenting', 'Society']

tech_ids = (
    data_exploded_df
    .query("type == 'Technology'")
    .query("year >= 2013")
    .drop_duplicates('id')
    .id.to_list()
)

type_ids = data_exploded_df.query("type in @check_types").id.to_list()

# Data not tagged with the main application types
_data_df = (
    data_df
    .query("id in @tech_ids and ~(id in @type_ids)")
)

# include also patents mentioning health or medical
adjust_ids = _data_df[_data_df.text.str.contains('health|medical', case=False)].id.to_list()
data_df.loc[data_df.id.isin(adjust_ids), 'topics'] = data_df.loc[data_df.id.isin(adjust_ids), 'topics'] + ', health'

# include also patents mentioning education
adjust_ids = _data_df[_data_df.text.str.contains('education', case=False)].id.to_list()
data_df.loc[data_df.id.isin(adjust_ids), 'topics'] = data_df.loc[data_df.id.isin(adjust_ids), 'topics'] + ', cognitive'

# include also patents mentioning education
adjust_ids = _data_df[_data_df.text.str.contains('parenting|parent', case=False)].id.to_list()
data_df.loc[data_df.id.isin(adjust_ids), 'topics'] = data_df.loc[data_df.id.isin(adjust_ids), 'topics'] + ', parenting2'

# preschool
adjust_ids = _data_df[_data_df.text.str.contains('kindergarten', case=False)].id.to_list()
data_df.loc[data_df.id.isin(adjust_ids), 'topics'] = data_df.loc[data_df.id.isin(adjust_ids), 'topics'] + ', preschool'

In [389]:
adjust_ids = data_df[data_df.text.str.contains('internet|online', case=False)].id.to_list()
remove_ids = data_df[data_df.text.str.contains('internet of things', case=False)].id.to_list()
data_df.query("id in @adjust_ids and ~(id in @remove_ids)").sample(10)

,id,text,dataset,topics,year,country_code
1567,CN-206282417-U,"A kind of urine-wet alarm device with historical record and forecast function. Utility model discloses a kind of urine-wet alarm device with historical record and forecast function, including defe...",patents,"ai2, infancy",2017,CN
19400,CN-103743433-A,Environmental online monitoring system for the production of formula food for infants and young children. The invention discloses an environment online monitoring system for producing infant formu...,patents,infancy,2014,CN
798,KR-20230107958-A,Electronic document creation and management system. The present invention relates to an electronic document creation and management system for teachers in child care institutions for infants and y...,patents,"preschool, infancy",2023,KR
1637,CN-207529479-U,Share the transmitting-receiving cabinet of mobile phone charge pal in a kind of internet. The utility model discloses the transmitting-receiving cabinet that mobile phone charge pal is shared in ...,patents,"mobile, infancy",2018,CN
23621,KR-102608276-B1,"Languange learning method for child. A method of learning language for infants using audio-visual content for learning and letter block materials for learning, in which primary learning of the wor...",patents,"mobile, communication, infancy",2023,KR
2205,US-2014330608-A1,"Scheduling and payment systems and methods. The present systems and/or methods generally facilitate scheduling and/or payment of a service provider. In one embodiment, the scheduling system is int...",patents,NaN,2014,US
1333,KR-102181667-B1,"System for supporting customer to serve dispatchment of mother helper and childrearing utilizing ict device. The present invention relates to an online customer support system, and more specifical...",patents,"parenting2, infancy",2020,KR
3523,CA-3153673-A1,Vital sign data management system and method. To provide a vital sign data management system and method that share vital sign data of children in real time between a facility side such as a nurser...,patents,"ai2, health",2022,CA
3530,KR-20190112374-A,A contact type memory card including a surface for printing an embedded content image and having pads in mutual contact with the connectors in the lateral and longitudinal directions. In the curre...,patents,"mobile, cognitive",2019,KR
5805,US-2014346207-A1,"STROLLER STREAMER SPONGE CARRIER, hypoallergenic carrier, holder for mobile streaming devices such as iPod, iPad, iPhone,iPad Mini, Kindle, Nook, and all or any mobile device able to stream Intern...",patents,"mobile, infancy, parenting2",2014,US


In [337]:
# Transform to one id and topic pair per row
data_exploded_df = utils.explode_data(data_df).query("topic != 'arts'")

### Patents with not application areas

In [364]:
type_ids = data_exploded_df.query("type in @check_types").id.to_list()

# Data not tagged with the main application types
_data_df = (
    data_df
    .query("id in @tech_ids and ~(id in @type_ids)")
)

In [368]:
_data_df.sample(20)

,id,text,dataset,topics,year,country_code
6915,CN-107808658-A,Real-time baby audio series behavior detection method based on home environment. The invention proposes a real-time baby audio series behavior detection method based on the home environment. First...,patents,"ai2, infancy",2018,CN
5058,CN-207117820-U,"A video transmission baby monitoring system. The utility model relates to the technical field of baby monitoring systems, and in particular discloses a video transmission baby monitoring system, w...",patents,"mobile, infancy",2018,CN
3174,KR-101341589-B1,User automatic recognition system using Zigbee tag. The present invention provides a user automatic recognition system using a Zigbee tag. The automatic user recognition system using the Zigbee ta...,patents,"ai2, infancy",2013,KR
21489,WO-2017000292-A1,"Smart tracking of baby feeding habits. A solution is provided for monitoring and analyzing baby feeding habits anywhere and anytime. The baby feeding habits data, e.g., feeding times, types of con...",patents,"ai2, mobile, infancy",2017,WO
4143,CN-108266007-A,A kind of bus platform that paper nappy is replaced convenient for baby based on Internet of Things. The present invention relates to a kind of bus platforms that paper nappy is replaced convenien...,patents,"wearables, infancy",2018,CN
14096,CN-103810816-A,"A baby vomit monitor. The invention relates to a baby vomit monitor, which comprises a wearing piece, a humidity sensor, a control unit and an early warning unit. The wearing piece is arranged bel...",patents,"ai2, infancy",2014,CN
8092,KR-101899865-B1,Study system using smart cube. The present invention discloses a learning system using a smart cube capable of actively inducing language learning of people who want to learn Korean or English inc...,patents,"ai2, infancy",2018,KR
16946,US-2013342691-A1,Infant monitoring systems and methods using thermal imaging. Various techniques are disclosed for systems and methods using thermal imaging to monitor an infant or other persons that may need obse...,patents,"ai2, infancy",2013,US
14288,CN-111879373-A,"An intelligent detection system for diapers. The invention provides an intelligent detection system for diapers, the system includes a tension detection module and an electric control device, the ...",patents,"ai2, infancy",2020,CN
21198,CN-110661921-A,A mobile phone remote control smart baby bottle based on Internet of Things technology. The invention discloses a mobile phone remote control intelligent milk bottle based on the Internet of Thing...,patents,"wearables, mobile, infancy",2020,CN


For example: Monitoring and snesor systems (without clear reason for monitoring), intelligent devices (eg, ceiling fan, anti-theft baby clothes), behaviour detection and recognition, general infant care systems, baby vomit monitor etc

## Baseline trends

Baseline trends for patent counts

In [338]:
importlib.reload(utils);
baseline_df = utils.get_baseline_patents()

In [339]:
trends_baseline = au.ts_magnitude_growth_(
    ts_df = baseline_df,
    year_start = 2019,
    year_end = 2023  
)
trends_baseline

,magnitude,growth
counts,7572871.2,31.192759


In [340]:
fig = pu.ts_smooth(
    baseline_df.assign(Total="Total").assign(counts = lambda df: df.counts/1e+6),
    ["Total"],
    variable= "counts",
    variable_title = "Publications (millions)",
    category_column = "Total",
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig)

alt.Chart(...)

## Insight 0: Overall trends

Early-years project growth of funding and project counts trends



In [369]:
ts_counts = utils.get_timeseries(data_df.drop_duplicates('id'), column='id')

In [370]:
ts_counts

,year,counts
0,2013,1046
1,2014,1301
2,2015,2056
3,2016,2107
4,2017,2288
5,2018,2802
6,2019,2937
7,2020,3065
8,2021,3509
9,2022,2751


In [371]:
au.ts_magnitude_growth_(
    ts_df = ts_counts,
    year_start = 2019,
    year_end = 2023  
)

,magnitude,growth
counts,2874.4,4.273078


In [372]:
fig = pu.ts_smooth(
    ts_counts.assign(Total="Total"),
    ["Total"],
    variable= "counts",
    variable_title = "Publications",
    category_column = "Total",
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig)

alt.Chart(...)

In [373]:
utils.get_data_distribution(data_exploded_df.query("year >= 2019"), column='type', values=['id'])

,type,counts,counts_prop
0,Biosciences,275,0.019
1,Child care & preschool,619,0.043
2,Development & learning,767,0.053
3,General,10587,0.737
4,Health,3490,0.243
5,Parenting,277,0.019
6,Society,31,0.002
7,Technology,2618,0.182


In [346]:
importlib.reload(utils);
ts_df = (
    utils.get_data_distribution(data_exploded_df, column='type', values=['id'], ts=True)
    .query("type != 'General'")
)
utils.get_data_magnitude_growth(data_exploded_df, ids=None, column='type', value='id')

,magnitude,growth,type,counts
6,6.2,175.000000,Society,31
1,123.8,28.771930,Child care & preschool,619
2,153.4,25.360231,Development & learning,767
0,55.0,20.833333,Biosciences,275
4,698.0,9.443861,Health,3490
3,2117.4,6.734238,General,10587
7,523.6,-5.220126,Technology,2618
5,55.4,-25.136612,Parenting,277


In [347]:
fig = pu.ts_smooth(
    ts_df,
    ts_df['type'].unique(),
    variable= "counts",
    variable_title = "",
    category_column = 'type',
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 

alt.Chart(...)

## Insight 1: Technology trends

- Magnitude and growth for technology topic overall
- Distribution of different technologies
- Growth of different technologies in UKRI funding


### Overall technology topic growth

In [374]:
tech_subtypes = set(topics_df.query("type == 'Technology'").subtype.unique())
tech_subtypes

{'AI', 'Immersive tech', 'Internet', 'Mobile'}

In [375]:
tech_type_df = (
    data_exploded_df
    # All items that belong to the allowed technology subtypes
    .query('subtype in @tech_subtypes')
    # Drop all duplicates that belong to technology type
    .drop_duplicates(['id', 'type'])
)

In [376]:
# ts_amounts_tech = utils.get_timeseries(tech_type_df, column='amount')
ts_counts_tech = utils.get_timeseries(tech_type_df, column='id')
utils.plot_quick_ts(ts_counts_tech, 'counts')

alt.Chart(...)

In [377]:
au.ts_magnitude_growth_(ts_counts_tech, year_start = 2019, year_end = 2023)

,magnitude,growth
counts,523.6,-5.220126


### Distribution of different technologies

In [378]:
tech_subtype_df = (
    data_exploded_df
    # All items that belong to the allowed technology subtypes
    .query('subtype in @tech_subtypes')
    .query("type == 'Technology'")
    # Drop all duplicates that belong to technology type
    .drop_duplicates(['id', 'subtype'])
)

In [379]:
# Total tech funding
counts_total = tech_subtype_df.drop_duplicates('id').query("year >= 2019").id.nunique()

In [380]:
tech_subtype_dist = (
    tech_subtype_df
    .query("year >= 2019")
    .groupby('subtype')
    .agg(
        counts=('id', 'nunique'), 
    )
    .reset_index()
    .assign(counts_prop = lambda df: df.counts/counts_total)
)

tech_subtype_dist

,subtype,counts,counts_prop
0,AI,1637,0.625286
1,Immersive tech,874,0.333843
2,Internet,10,0.003820
3,Mobile,677,0.258594


### Growth of technology topics

In [355]:
column = 'subtype'
value = 'counts'

tech_subtype_ts = (
    tech_subtype_df
    .drop_duplicates(['id', column])
    .groupby(['subtype', 'year'])
    .agg(
        counts=('id', 'nunique'), 
    )
    .reset_index()
)

tech_subtype_ts = utils.impute_empty_periods_all_ts(tech_subtype_ts, column)

utils.magnitude_and_growth(tech_subtype_ts, column, value)

,magnitude,growth,subtype
0,327.4,9.630459,AI
0,174.8,-14.874552,Immersive tech
0,2.0,-33.333333,Internet
0,135.4,-42.229730,Mobile


In [356]:
fig = pu.ts_smooth(
    tech_subtype_ts,
    ["AI", "Immersive tech", "Internet", "Mobile"],
    variable= "counts",
    variable_title = "",
    category_column = column,
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 

alt.Chart(...)

## Insight 2: Applications

- Where are these technologies applied the most?
- Where do we see growth vs stagnation when it comes to applications?

In [390]:
tech_ids = (
    data_exploded_df
    .query("type == 'Technology'")
    .query("year >= 2013")
    .drop_duplicates('id')
    .id.to_list()
)

tech_ids_5y = (
    data_exploded_df
    .query("type == 'Technology'")
    .query("year >= 2019")
    .drop_duplicates('id')
    .id.to_list()
)

In [391]:
len(tech_ids_5y)

2618

### Application distribution

In [392]:
column = 'type'

tech_applications_df = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids_5y'), 
    column=column, 
    values=['id']
)


tech_applications_ts = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids').query("type != 'Technology' and type != 'General'"),
    column=column, 
    values=['id'],
    ts=True
)



In [393]:
tech_applications_df

,type,counts,counts_prop
0,Biosciences,85,0.032
1,Child care & preschool,173,0.066
2,Development & learning,260,0.099
3,General,2055,0.785
4,Health,920,0.351
5,Parenting,261,0.1
6,Society,2,0.001
7,Technology,2618,1.0


In [394]:
fig = pu.ts_smooth(
    tech_applications_ts,
    tech_applications_ts[column].unique(),
    variable= "counts",
    variable_title = "",
    category_column = column,
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 

alt.Chart(...)

In [395]:
utils.get_data_magnitude_growth(data_exploded_df, ids=tech_ids, column=column, value='id')

,magnitude,growth,type,counts
7,0.4,inf,Society,2
6,17.0,177.272727,Biosciences,85
3,184.0,5.482042,Health,920
1,52.0,0.714286,Development & learning,260
2,411.0,-0.165289,General,2055
5,523.6,-5.220126,Technology,2618
0,34.6,-23.622047,Child care & preschool,173
4,52.2,-28.491620,Parenting,261


In [397]:
trends_df = utils.get_data_magnitude_growth(data_exploded_df, ids=tech_ids, column=column, value='id')
(
    tech_applications_df
    .merge(trends_df.drop('counts', axis=1), on='type')[['type', 'magnitude', 'growth', 'counts', 'counts_prop']]
    .query("type != 'Technology' and type != 'General'")
)

,type,magnitude,growth,counts,counts_prop
0,Biosciences,17.0,177.272727,85,0.032
1,Child care & preschool,34.6,-23.622047,173,0.066
2,Development & learning,52.0,0.714286,260,0.099
4,Health,184.0,5.482042,920,0.351
5,Parenting,52.2,-28.491620,261,0.1
6,Society,0.4,inf,2,0.001


In [399]:
from discovery_child_development.utils import chart_trends
importlib.reload(chart_trends);

chart_trends.estimate_trend_type(
    trends_df.query("type != 'Technology' and type != 'General'"), 
    magnitude_column='magnitude', 
    growth_column='growth'
)

,magnitude,growth,type,counts,trend_type_suggestion
7,0.4,inf,Society,2,emerging
6,17.0,177.272727,Biosciences,85,emerging
3,184.0,5.482042,Health,920,hot
1,52.0,0.714286,Development & learning,260,hot
0,34.6,-23.622047,Child care & preschool,173,dormant
4,52.2,-28.491620,Parenting,261,stable


In [401]:
# scatter chart of trends_df
import altair as alt
alt.Chart(
    trends_df.query("type != 'Technology' and type != 'General'")
).mark_point().encode(
    x='magnitude:Q',
    y='growth:Q',
    color='type:N',
    tooltip=['type', 'magnitude', 'growth']
)


alt.Chart(...)

In [178]:
# pd.set_option('display.max_colwidth', 200)
# (
#     data_exploded_df
#     .query('id in @tech_ids')
#     # .query("subtype == 'Personal social emotional'")
#     .query("type == 'Social'")
#     .drop_duplicates(['id'])
#     .sort_values('year', ascending=False)
# )[['id', 'text', 'topics', 'year']]

### Application distribution: More granular subtypes

In [310]:
column = 'subtype'

tech_applications_df = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids_5y'), 
    column=column, 
    values=['id']
)


tech_applications_ts = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids').query("type != 'Technology'"),
    column=column, 
    values=['id'],
    ts=True
)
tech_applications_df.query("type != 'Technology'").sort_values('counts', ascending=False)

,subtype,counts,counts_prop,type
8,Infancy,1979,0.756,General
24,Sleep,430,0.164,Health
6,Health,321,0.123,Health
23,Preschool,162,0.062,Child care & preschool
21,Physical development,161,0.061,Health
2,Cognitive development,158,0.06,Development & learning
4,Games,97,0.037,General
14,Neuroscience,84,0.032,Biosciences
22,Prenatal,43,0.016,Health
26,Special educational needs,37,0.014,Development & learning


In [123]:
(
    utils.get_data_magnitude_growth(data_exploded_df, ids=tech_ids, column=column, value='id')
    .sort_values(['type', 'growth'], ascending=False)
    # .sort_values('growth', ascending=False)
)

,magnitude,growth,subtype,counts,type
12,327.4,9.630459,AI,1637,Technology
19,174.8,-14.874552,Immersive tech,874,Technology
22,2.0,-33.333333,Internet,10,Technology
24,135.4,-42.229730,Mobile,677,Technology
0,0.4,inf,Social services,2,Social
9,1.0,50.000000,Parenting,5,Parenting
5,2.4,125.000000,Nutrition & weight,12,Health
8,8.6,75.000000,Prenatal,43,Health
10,32.2,17.283951,Physical development,161,Health
11,24.2,16.666667,Health,121,Health


In [37]:
cat_type = 'Development & learning'
cats = list(topics_df.query("type == @cat_type").subtype.unique())

In [38]:
fig = pu.ts_smooth(
    tech_applications_ts,
    cats,
    variable= "counts",
    variable_title = "",
    category_column = column,
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 


alt.Chart(...)

### Which technology is applied to most to subtype X?

## Insight 3: Geographical insights

- Top countries in terms of counts
- UK vs baseline growth for overall counts, in technology counts and application counts

In [205]:
# data_exploded_df.explode('country_code').drop_duplicates(['id', 'country_code']).isnull().sum()
# Consider lack of information

In [290]:
data_countries_df = (
    data_exploded_df
    # .explode('country_code')
    .dropna(subset=['country_code'])
    .query("type == 'Technology'")
    .query('subtype in @tech_subtypes')
    .drop_duplicates(['id'])    
)
country_codes = data_countries_df.country_code.unique()

growth_df = []
ts_counts = []
for country_code in country_codes:
    country_df = data_countries_df.query("country_code == @country_code")
    _ts_counts = utils.get_timeseries(country_df, column='id').assign(country_code = country_code)
    growth_df.append(
        au.ts_magnitude_growth_(
            ts_df = _ts_counts,
            year_start = 2019,
            year_end = 2023  
        )
        .assign(country_code = country_code)
        .reset_index(drop=True)
    )
    ts_counts.append(_ts_counts)
growth_df = pd.concat(growth_df, ignore_index=True)
ts_counts = pd.concat(ts_counts, ignore_index=True)

In [291]:
(
    growth_df
    .sort_values('magnitude', ascending=False)
    .head(20)
)

,magnitude,growth,country_code
0,334.6,-15.964126,CN
1,80.0,32.596685,KR
2,36.6,-13.274336,US
6,20.8,-9.836066,WO
11,12.2,11.428571,JP
4,7.8,141.666667,EP
5,6.4,69.230769,TW
3,4.2,800.000000,AU
9,2.8,120.000000,TR
7,2.8,125.000000,CA


In [292]:
countries = ['US', 'GB', 'CN', 'KR', 'JP']
fig = pu.ts_smooth(
    ts_counts.query("country_code in @countries"),
    countries,
    variable= "counts",
    variable_title = "",
    category_column = 'country_code',
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 


alt.Chart(...)

In [42]:
data_countries_df = (
    data_exploded_df
    .explode('country_code')
    .dropna(subset=['country_code'])
    .query("type == 'Technology'")
    .query('subtype in @tech_subtypes')
    .drop_duplicates(['id', 'subtype']) 
    .query("country_code == 'GB'")   
)

In [43]:
data_countries_df.groupby('subtype').agg(counts=('id', 'nunique')).reset_index()

,subtype,counts
0,AI,9
1,Immersive tech,3
2,Mobile,10


## Check detailed applications

In [150]:
importlib.reload(utils);
df = utils.get_counts_by_application(data_exploded_df, topics_df)
df.to_csv(utils.PROJECT_DIR / 'outputs/data/tables/tech_applications_patents.csv', index=False)

In [151]:
df.Total.sum()

1144

## Export tables

In [130]:
def check_if_nan_only_list_elements(x):
    return all([pd.isna(i) for i in x])

def convert_topic_columns(df):
    for col in ['topic', 'minor_category', 'major_category', 'topic_code']:
        df = df.assign(**{col : lambda df: df[col].apply(lambda x: list(x) if check_if_nan_only_list_elements(x)==False else [])})
        df = df.assign(**{col : lambda df: df[col].apply(lambda x: ", ".join([xx for xx in x if isinstance(xx, str)]) if len(x)>0 else "")})
    return df

In [131]:
_export_df = (
    data_df
    .fillna({'topics': ''})
    .assign(topics = lambda df: df['topics'].apply(lambda x: [t.strip() for t in x.split(',') if isinstance(t, str)]))
    .explode('topics')
    .merge(topics_df, left_on='topics', right_on='topic', how='left')
    .rename(columns={'topic': 'topic_code', 'type': 'major_category', 'subtype': 'minor_category', 'name': 'topic'})
    .drop('topics', axis=1)
)  
export_df = (
    data_df[['id', 'text', 'dataset', 'year', 'country_code']]
    .merge(
        _export_df[['id', 'topic', 'major_category', 'minor_category', 'topic_code']].groupby('id').agg(set),
        on='id',
        how='left'
    )
    .pipe(convert_topic_columns)
    .assign(url = lambda df: "https://patents.google.com/patent/" + df['id'].str.replace("-", ""))
)

In [132]:
export_df.to_csv(utils.PROJECT_DIR / "outputs/data/tables/patents_final.csv", index=False)

In [297]:
data_exploded_df.query("topic == 'parenting2'").sample(10)

,id,text,dataset,topics,year,country_code,topic,type,subtype,name
5644,EP-4064695-A1,"Monitoring system. A control system is provided for controlling communication between a monitor unit and a set of remote user units. A monitoring system including the control system, the monitor u...",patents,parenting2,2022,EP,parenting2,Parenting,Parenting,Parenting
5166,KR-101781031-B1,"Alarm for condition based on IoT. The present invention relates to a diaper, and more particularly, to an IoT-based status alarm device that accurately detects a subject by temperature, humidity, ...",patents,parenting2,2017,KR,parenting2,Parenting,Parenting,Parenting
2099,CN-210516219-U,"Smart Baby Monitor. The utility model relates to an intelligent baby monitor, which comprises a main body and a base. The main body is provided with a communication module, a microphone and a spea...",patents,parenting2,2020,CN,parenting2,Parenting,Parenting,Parenting
10802,CN-106510664-A,"Wristband for infants. The invention relates to a wristband for infants. The wristband comprises a wristband body. The wristband body comprises a body temperature measuring unit, a pulse measuring...",patents,parenting2,2017,CN,parenting2,Parenting,Parenting,Parenting
7970,CN-210248684-U,"Anti-theft intelligent ring wrist strap. The utility model discloses an intelligence of preventing burglary encircles wrist strap, including upper strata membrane, lower rete and intermediate leve...",patents,parenting2,2020,CN,parenting2,Parenting,Parenting,Parenting
36241,US-2013123658-A1,"Child-Care Robot and a Method of Controlling the Robot. A child-care robot for use in a nursery school associates child behavior patterns with corresponding robot action patterns, and acquires a c...",patents,parenting2,2013,US,parenting2,Parenting,Parenting,Parenting
21771,CN-207637203-U,"Kindergarten parent registers video machine. It registers video machine the utility model discloses kindergarten parent,Including video machine of registering,Built-in control circuit system is eq...",patents,parenting2,2018,CN,parenting2,Parenting,Parenting,Parenting
35189,WO-2019170921-A1,A comfort apparatus for a mother and an infant. A comfort apparatus (1) for use by a parent and an infant. The comfort apparatus (1) has a parent support and comfort means (4) and an infant portio...,patents,parenting2,2019,WO,parenting2,Parenting,Parenting,Parenting
42398,CN-211184123-U,"a baby surveillance camera. The utility model relates to the technical field of monitors, in particular to a baby monitoring camera, comprising a base and a camera body arranged on the base; a con...",patents,parenting2,2020,CN,parenting2,Parenting,Parenting,Parenting
996,CN-110141419-A,A wearable physical cooling clothing for children. The invention discloses a wearable physical cooling clothing for children. The cooling headband includes a head built-in water pipe and a head te...,patents,parenting2,2019,CN,parenting2,Parenting,Parenting,Parenting
